# Graph Neural Network for OMR Link Prediction

This notebook implements a **Graph Neural Network (GNN)** approach for predicting edges between music notation symbols, replacing the simple MLP architecture.

## Why GNN vs MLP?

### Current MLP Approach
- **Input**: Two isolated nodes (bounding boxes + class labels)
- **Process**: Concatenate features → MLP layers → Binary prediction
- **Limitation**: No understanding of graph structure or neighborhood context

### GNN Approach
- **Input**: Entire notation graph with all nodes and their relationships
- **Process**: Message passing between nodes → Aggregate neighborhood information → Predict edges
- **Advantages**:
  - Captures **global graph structure** and dependencies
  - Learns from **neighborhood context** (e.g., noteheads near the same stem)
  - Better handles **long-range dependencies** (e.g., beams connecting multiple notes)
  - More suited for **structured prediction** in music notation assembly

---

## Implementation Overview

We'll transition from:
```
MLP: (node_i, node_j) → edge_probability
```

To:
```
GNN: Graph(all_nodes, partial_edges) → edge_probabilities_for_all_pairs
```

## Step 11: Next Steps and Improvements

### What You've Accomplished
✅ Converted your OMR link prediction from MLP to GNN  
✅ Implemented message passing for graph-structured data  
✅ Trained GAT-based model for edge prediction  
✅ Evaluated on MUSCIMA++ dataset  

### Potential Improvements

1. **Try Different GNN Architectures**
   - GraphSAGE: Better for large graphs
   - GAT with more attention heads
   - Deeper networks (more layers)

2. **Advanced Features**
   - Add positional encodings (relative positions between nodes)
   - Include edge features (distance, angle between nodes)
   - Multi-task learning (predict edge type + existence)

3. **Handle Class Imbalance**
   - Adjust `POS_WEIGHT` parameter
   - Use focal loss instead of BCE loss
   - Sample negative edges more carefully

4. **Optimize Performance**
   - Increase batch size if memory allows
   - Use learning rate scheduling
   - Add batch normalization or layer normalization

5. **Graph Construction Strategy**
   - Currently using all pairs or distance-filtered pairs
   - Try k-nearest neighbors graph
   - Use learned edge probabilities to refine graph structure iteratively

6. **Visualization**
   - Visualize attention weights (for GAT)
   - Plot predicted vs ground truth graphs
   - Analyze failure cases

### Code to Save GNN Model for Inference

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import yaml
import os
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt

import torch_geometric
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GCNConv, GATConv, SAGEConv, global_mean_pool
from torch_geometric.utils import negative_sampling

from utils.data_pool_gt import load_ground_truth_data
from utils.constants import get_classlist_and_classdict
from utils.utility import set_seed
from configs.assembler.default import get_cfg_defaults

print(f'PyTorch version: {torch.__version__}')
print(f'PyTorch Geometric version: {torch_geometric.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')

PyTorch version: 2.8.0
PyTorch Geometric version: 2.7.0
CUDA available: False


In [53]:
class GNN_LinkPredictor(nn.Module):
    """
    Graph Neural Network for music notation link prediction.
    
    Architecture:
    1. Node feature encoding (bbox + class embedding)
    2. Graph convolution layers (message passing)
    3. Edge prediction head (concatenate node pairs → MLP → probability)
    """
    
    def __init__(self, vocab_size, embedding_dim=32, hidden_dim=64, num_layers=3, dropout=0.2):
        super(GNN_LinkPredictor, self).__init__()
        
        # Node feature encoding
        self.class_embedding = nn.Embedding(vocab_size, embedding_dim)
        self.node_encoder = nn.Linear(4 + embedding_dim, hidden_dim)  # bbox (4) + class embedding
        
        # Graph convolution layers (using GAT for attention-based message passing)
        self.convs = nn.ModuleList()
        self.convs.append(GATConv(hidden_dim, hidden_dim, heads=4, concat=False, dropout=dropout))
        
        for _ in range(num_layers - 1):
            self.convs.append(GATConv(hidden_dim, hidden_dim, heads=4, concat=False, dropout=dropout))
        
        # Edge prediction head
        self.edge_mlp = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1)
        )
        
        self.dropout = dropout
    
    def encode_nodes(self, x, edge_index):
        """
        Encode node features through GNN layers.
        
        Args:
            x: Node features [num_nodes, feature_dim]
            edge_index: Edge connectivity [2, num_edges]
        
        Returns:
            Node embeddings [num_nodes, hidden_dim]
        """
        # Initial node encoding
        h = F.relu(self.node_encoder(x))
        h = F.dropout(h, p=self.dropout, training=self.training)
        
        # Message passing through GNN layers
        for conv in self.convs:
            h = conv(h, edge_index)
            h = F.relu(h)
            h = F.dropout(h, p=self.dropout, training=self.training)
        
        return h
    
    def decode_edges(self, z, edge_label_index):
        """
        Predict edge probabilities from node embeddings.
        
        Args:
            z: Node embeddings [num_nodes, hidden_dim]
            edge_label_index: Edges to predict [2, num_edge_predictions]
        
        Returns:
            Edge logits [num_edge_predictions, 1]
        """
        # Get source and target node embeddings
        src = z[edge_label_index[0]]
        dst = z[edge_label_index[1]]
        
        # Concatenate and pass through MLP
        edge_features = torch.cat([src, dst], dim=-1)
        return self.edge_mlp(edge_features)
    
    def forward(self, data):
        """
        Forward pass for link prediction.
        Args:
            data: PyG Data object with:
                - x: Node features [num_nodes, feature_dim]
                - edge_index: Graph connectivity [2, num_edges]
                - edge_label_index: Edges to predict [2, num_predictions]
        Returns:
            Edge prediction logits [num_predictions, 1]
        """
        # Encode nodes through GNN
        z = self.encode_nodes(data.x, data.edge_index)
        # Decode edge predictions
        out = self.decode_edges(z, data.edge_label_index)
        return out

#GCN-based model (simpler, faster)
class GCN_LinkPredictor(nn.Module):
    """Simpler GCN-based link predictor (alternative to GAT)."""
    def __init__(self, vocab_size, embedding_dim=128, hidden_dim=128, num_layers=3, dropout=0.5):
        super(GCN_LinkPredictor, self).__init__()
        self.class_embedding = nn.Embedding(vocab_size, embedding_dim)
        self.node_encoder = nn.Linear(4 + embedding_dim, hidden_dim)
        self.convs = nn.ModuleList()
        self.convs.append(GCNConv(hidden_dim, hidden_dim))
        for _ in range(num_layers - 1):
            self.convs.append(GCNConv(hidden_dim, hidden_dim))
        self.edge_mlp = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1)
        )
        self.dropout = dropout
    
    def encode_nodes(self, x, edge_index):
        h = F.relu(self.node_encoder(x))
        for conv in self.convs:
            h = conv(h, edge_index)
            h = F.relu(h)
            h = F.dropout(h, p=self.dropout, training=self.training)
        return h
    
    def decode_edges(self, z, edge_label_index):
        src = z[edge_label_index[0]]
        dst = z[edge_label_index[1]]
        edge_features = torch.cat([src, dst], dim=-1)
        return self.edge_mlp(edge_features)
    
    def forward(self, data):
        z = self.encode_nodes(data.x, data.edge_index)
        return self.decode_edges(z, data.edge_label_index)

print("GNN models defined!")
print("- GNN_LinkPredictor: GAT-based (attention mechanism)")
print("- GCN_LinkPredictor: GCN-based (simpler, faster)")

GNN models defined!
- GNN_LinkPredictor: GAT-based (attention mechanism)
- GCN_LinkPredictor: GCN-based (simpler, faster)


In [74]:
class MUSCIMAGraphDataset(torch.utils.data.Dataset):
    """
    Convert MUSCIMA++ ground truth to PyG graph format.
    
    Each graph represents one music score page with:
    - Nodes: Music notation objects (noteheads, stems, etc.)
    - Edges: Structural connections between objects
    """
    def __init__(self, data_pool, class_dict, normalize_bbox=True):
        """
        Args:
            data_pool: GroundTruthDataPool instance
            class_dict: Dictionary mapping class names to IDs
            normalize_bbox: Whether to normalize bounding boxes (deprecated - normalization done in data_pool)
        """
        self.data_pool = data_pool
        self.class_dict = class_dict
        #get inference graph (bboxes already normalized if data_pool.normalize_bboxes=True)
        self.graphs = data_pool.get_inference_graph()
        self.graph_indices = list(self.graphs.keys())
        
        print(f"Created {len(self.graph_indices)} graph objects")
    
    def __len__(self):
        return len(self.graph_indices)
    
    def __getitem__(self, idx):
        """
        Convert one document to a PyG Data object.
        
        Returns:
            PyG Data with:
            - x_bbox: Node bounding boxes [num_nodes, 4] (already normalized)
            - x_class: Node class indices [num_nodes]
            - edge_index: Graph connectivity (positive edges from ground truth)
            - edge_label_index: All edges to predict (pos + neg)
            - edge_label: Ground truth labels (1=connected, 0=not connected)
        """
        graph_idx = self.graph_indices[idx]
        inference_data = self.graphs[graph_idx]
        mung = self.data_pool.mungs[graph_idx]
        
        # Use pre-computed normalized bboxes from get_inference_graph()
        # source_bbox and target_bbox contain all pairwise combinations
        # We need to extract unique node bboxes
        
        # Get unique nodes and their features
        nodes = mung.vertices
        num_nodes = len(nodes)
        
        # Create node ID to index mapping
        node_id_to_idx = {node.id: i for i, node in enumerate(nodes)}
        
        # Extract node bounding boxes and classes
        # Use the already-normalized bboxes from inference_data
        x_bbox = inference_data['source_bbox'][:num_nodes]  # First num_nodes entries are unique nodes
        
        # Extract node classes
        node_classes = []
        for node in nodes:
            node_classes.append(self.class_dict[node.class_name])
        
        x_class = torch.tensor(node_classes, dtype=torch.long)  # [num_nodes]
        
        # Build positive edges from ground truth (edge_index)
        pos_edges = []
        for node in nodes:
            src_idx = node_id_to_idx[node.id]
            for target_id in node.outlinks:
                if target_id in node_id_to_idx:
                    tgt_idx = node_id_to_idx[target_id]
                    pos_edges.append([src_idx, tgt_idx])
        
        if len(pos_edges) > 0:
            edge_index = torch.tensor(pos_edges, dtype=torch.long).t()
        else:
            # If no positive edges, create empty edge_index
            edge_index = torch.zeros((2, 0), dtype=torch.long)
        
        # Extract edge_label_index and edge_label for all candidate pairs
        # IMPORTANT: Convert MUNG node IDs to sequential indices for PyG batching
        source_ids = inference_data['source_id'].numpy()
        target_ids = inference_data['target_id'].numpy()
        
        # Map MUNG IDs to sequential indices using node_id_to_idx
        source_indices = torch.tensor([node_id_to_idx[int(sid)] for sid in source_ids], dtype=torch.long)
        target_indices = torch.tensor([node_id_to_idx[int(tid)] for tid in target_ids], dtype=torch.long)
        
        edge_label_index = torch.stack([source_indices, target_indices], dim=0)
        edge_label = inference_data['label'].squeeze(-1)
        # Create PyG Data object
        data = Data(
            x_bbox=x_bbox,
            x_class=x_class,
            edge_index=edge_index,
            edge_label_index=edge_label_index,
            edge_label=edge_label,
            num_nodes=num_nodes
        )
        return data

def collate_graph_batch(batch):
    """
    Custom collate function to batch multiple graphs.
    PyG's Batch.from_data_list handles graph batching automatically.
    """
    return Batch.from_data_list(batch)


print("Graph dataset class defined!")
print("Key components:")
print("- Converts MUSCIMA++ annotations to PyG graphs")
print("- Uses pre-normalized bboxes from get_inference_graph()")
print("- Extracts node features (bbox + class)")
print("- Builds edge_index from ground truth connections")
print("- Prepares edge_label_index for all pairs to predict")

Graph dataset class defined!
Key components:
- Converts MUSCIMA++ annotations to PyG graphs
- Uses pre-normalized bboxes from get_inference_graph()
- Extracts node features (bbox + class)
- Builds edge_index from ground truth connections
- Prepares edge_label_index for all pairs to predict


In [54]:
# Experiment configuration
exp_name = 'gnn_training_v1'
output_dir = 'outputs'

# Data paths (same as MLP)
gt_annotations_root = 'data/MUSCIMA++/v2.0/data/annotations'
images_root = 'data/MUSCIMA++/datasets_r_staff/images'
split_file = 'splits/mob_split.yaml'
classes = 'essential'

# Model hyperparameters
EMBEDDING_DIM = 128
HIDDEN_DIM = 128
NUM_GNN_LAYERS = 3
DROPOUT = 0.2

# Training hyperparameters
NUM_EPOCHS = 50
LEARNING_RATE = 1e-3
BATCH_SIZE = 4  # Smaller batch size for graph data (each batch contains multiple graphs)
POS_WEIGHT = 2.0  # Weight for positive class (adjust based on class imbalance)

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# Create output directory
os.makedirs(f'{output_dir}/{exp_name}', exist_ok=True)

# Set seed for reproducibility
set_seed(42)

print(f'\\nExperiment: {exp_name}')
print(f'Model: GAT-based GNN')
print(f'Hidden dim: {HIDDEN_DIM}, Layers: {NUM_GNN_LAYERS}')
print(f'Epochs: {NUM_EPOCHS}, Batch size: {BATCH_SIZE}, LR: {LEARNING_RATE}')

Using device: cpu
\nExperiment: gnn_training_v1
Model: GAT-based GNN
Hidden dim: 128, Layers: 3
Epochs: 50, Batch size: 4, LR: 0.001


In [96]:
%%capture --no-display
# Get class information
class_list, class_dict_original = get_classlist_and_classdict(classes)
class_list = list(class_list)

# Keep original class_dict for data loading (it needs ALL classes to compute exclude_classes)
vocab_size = len(class_list)

print(f'Loaded {vocab_size} classes')
print(f'Sample classes from original dict:')
for name in list(class_list)[:3]:
    print(f'  {name}: {class_dict_original[name]}')

# Load configuration
cfg = get_cfg_defaults()
with open(cfg.DATA.DATA_CONFIG, 'rb') as hdl:
    data_config = yaml.load(hdl, Loader=yaml.FullLoader)
data_config['mode'] = 'GNN'  # Set mode to GNN

print('\nLoading ground truth data...')
print('(Using original class_dict for filtering excluded classes)')
data_pools = load_ground_truth_data(
    gt_annotations_root=gt_annotations_root,
    images_root=images_root,
    split_file=split_file,
    class_list=class_list,
    class_dict=class_dict_original,  # Pass original dict for filtering
    config=data_config,
    load_training_data=True,
    load_validation_data=True,
    load_test_data=False
)

print(f'\nTraining documents: {len(data_pools["train"].mungs)}')
print(f'Validation documents: {len(data_pools["valid"].mungs)}')

# NOW re-index class_dict for embeddings (after data is loaded and filtered)
print('\n⚙️  Re-indexing class_dict to contiguous IDs for embeddings...')
class_dict = {name: idx for idx, name in enumerate(class_list)}
print(f'Re-indexed classes (0-{vocab_size-1}):')
for name in list(class_list)[:3]:
    print(f'  {name}: {class_dict[name]}')

# IMPORTANT: Update the data_pool's class_dict to the re-indexed version for BOTH train and valid
data_pools['train'].class_dict = class_dict
data_pools['valid'].class_dict = class_dict
print('\n✅ Data pool class_dict updated to re-indexed version (train & valid)')

# CRITICAL: Prepare training pairs before creating dataset
print('\n🔧 Preparing training entity pairs...')
data_pools['train'].prepare_train_entities()
print('✅ Training pairs prepared!')

print('\n🔧 Preparing validation entity pairs...')
data_pools['valid'].prepare_train_entities()
print('✅ Validation pairs prepared!')

In [ ]:
# %%capture --no-display
# # Get class information
# class_list, class_dict = get_classlist_and_classdict(classes)
# class_list = list(class_list)

# vocab_size = len(class_list)

# print(f'Loaded {vocab_size} classes with contiguous IDs')
# print(f'Classes: {list(class_dict.keys())[:10]}...')  # Show first 10

# # Load configuration
# cfg = get_cfg_defaults()
# with open(cfg.DATA.DATA_CONFIG, 'rb') as hdl:
#     data_config = yaml.load(hdl, Loader=yaml.FullLoader)
# data_config['mode'] = 'GNN'  # Set mode to GNN

# print('\nLoading ground truth data...')
# data_pools = load_ground_truth_data(
#     gt_annotations_root=gt_annotations_root,
#     images_root=images_root,
#     split_file=split_file,
#     class_list=class_list,
#     class_dict=class_dict,
#     config=data_config,
#     load_training_data=True,
#     load_validation_data=False,
#     load_test_data=False
# )

# print(f'\nTraining documents: {len(data_pools["train"].mungs)}')


In [97]:
# Convert to graph datasets
print('\nConverting to graph format...')

# Check if inference_graph contains tensor data (i.e., get_inference_graph() was called)
# If so, we need to call prepare_train_entities() again to reset to Node pairs
if len(data_pools['train'].inference_graph) > 0:
    sample_value = list(data_pools['train'].inference_graph.values())[0]
    if isinstance(sample_value, dict):  # Tensor format (from get_inference_graph())
        print('⚠️  Inference graph contains tensor data - re-preparing from scratch...')
        data_pools['train'].prepare_train_entities()
        print('✓ Training pairs re-prepared')

if len(data_pools['valid'].inference_graph) > 0:
    sample_value = list(data_pools['valid'].inference_graph.values())[0]
    if isinstance(sample_value, dict):  # Tensor format (from get_inference_graph())
        print('⚠️  Validation inference graph contains tensor data - re-preparing from scratch...')
        data_pools['valid'].prepare_train_entities()
        print('✓ Validation pairs re-prepared')

train_dataset = MUSCIMAGraphDataset(data_pools['train'], class_dict, normalize_bbox=True)
valid_dataset = MUSCIMAGraphDataset(data_pools['valid'], class_dict, normalize_bbox=True)
# test_dataset = MUSCIMAGraphDataset(data_pools['test'], class_dict, normalize_bbox=True)

# Create dataloaders
train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_graph_batch,
    num_workers=0
)

valid_loader = torch.utils.data.DataLoader(
    valid_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_graph_batch,
    num_workers=0
)

# test_loader = torch.utils.data.DataLoader(
#     test_dataset,
#     batch_size=BATCH_SIZE,
#     shuffle=False,
#     collate_fn=collate_graph_batch,
#     num_workers=0
# )

print(f'\nTrain loader: {len(train_loader)} batches')
print(f'Valid loader: {len(valid_loader)} batches')
# print(f'Test loader: {len(test_loader)} batches')


Converting to graph format...
[DEBUG] Preparing graph for inference...


Preparing inference graphs: 100%|██████████| 84/84 [00:15<00:00,  5.39it/s]



[DEBUG] Inference graph preparation completed in 15.58s
Created 84 graph objects
[DEBUG] Preparing graph for inference...


Preparing inference graphs: 100%|██████████| 28/28 [02:10<00:00,  4.67s/it]



[DEBUG] Inference graph preparation completed in 130.83s
Created 28 graph objects

Train loader: 21 batches
Valid loader: 7 batches
Created 28 graph objects

Train loader: 21 batches
Valid loader: 7 batches


## Reload Datasets with Fixed class_dict

Now we need to recreate the datasets with the corrected `class_dict`.

## Rebuild Model with Correct vocab_size

The model needs to be recreated with the corrected vocabulary size.

In [72]:
# Rebuild GNN model with corrected vocab_size
model = GCN_LinkPredictor(
    vocab_size=vocab_size,
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_GNN_LAYERS,
    dropout=DROPOUT
).to(device)

# Reinitialize optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(POS_WEIGHT).to(device))

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print('✅ Model rebuilt with correct vocab_size!')
print(f'Vocab size: {vocab_size}')
print(f'Embedding layer size: {model.class_embedding.num_embeddings}')
print(f'Total parameters: {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')

✅ Model rebuilt with correct vocab_size!
Vocab size: 73
Embedding layer size: 73
Total parameters: 108,929
Trainable parameters: 108,929


## Verify: Test One Batch

Let's verify the fix works by testing one batch.

In [43]:
from pdb import set_trace

In [92]:
# Test forward pass with one batch
print("Testing forward pass...")
model.eval()
with torch.no_grad():
    for batch in train_loader:
        batch = batch.to(device)
        
        # Check class indices
        print(f"Batch class indices: min={batch.x_class.min().item()}, max={batch.x_class.max().item()}")
        print(f"Expected range: [0, {vocab_size-1}]")
        
        # Prepare node features
        class_embed = model.class_embedding(batch.x_class)
        x = torch.cat([batch.x_bbox, class_embed], dim=-1)
        batch.x = x
        
        # Forward pass
        out = model(batch).squeeze()
        
        print(f"✅ Forward pass successful!")
        print(f"Output shape: {out.shape}")
        print(f"Sample predictions: {torch.sigmoid(out[:5])}")
        break

model.train()
print("\n🎉 All fixed! You can now proceed with training.")

Testing forward pass...
Batch class indices: min=0, max=72
Expected range: [0, 72]
✅ Forward pass successful!
Output shape: torch.Size([46470])
Sample predictions: tensor([0.5094, 0.5042, 0.5042, 0.5042, 0.5042])

🎉 All fixed! You can now proceed with training.


In [93]:
# Build GNN model
model = GNN_LinkPredictor(
    vocab_size=vocab_size,
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_GNN_LAYERS,
    dropout=DROPOUT
).to(device)

# Optimizer and loss
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(POS_WEIGHT).to(device))

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print('\\nModel built!')
print(f'Total parameters: {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')

\nModel built!
Total parameters: 267,649
Trainable parameters: 267,649


In [98]:
def train_epoch(model, loader, optimizer, criterion, device):
    """Train for one epoch."""
    model.train()
    total_loss = 0
    all_preds = []
    all_labels = []
    
    for batch in tqdm(loader, desc='Training'):
        # Move batch to device
        batch = batch.to(device)
        
        # Prepare node features (concatenate bbox + class embedding)
        class_embed = model.class_embedding(batch.x_class)
        x = torch.cat([batch.x_bbox, class_embed], dim=-1)
        
        # Create temporary data object with combined features
        batch.x = x
        
        # Forward pass
        optimizer.zero_grad()
        out = model(batch).squeeze()
        
        # Compute loss
        loss = criterion(out, batch.edge_label)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
        # Collect predictions
        with torch.no_grad():
            probs = torch.sigmoid(out)
            all_preds.extend(probs.cpu().numpy())
            all_labels.extend(batch.edge_label.cpu().numpy())
    
    # Calculate metrics
    avg_loss = total_loss / len(loader)
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    
    pred_binary = (all_preds > 0.5).astype(int)
    accuracy = (pred_binary == all_labels).mean()
    
    tp = ((pred_binary == 1) & (all_labels == 1)).sum()
    fp = ((pred_binary == 1) & (all_labels == 0)).sum()
    fn = ((pred_binary == 0) & (all_labels == 1)).sum()
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    
    return {
        'loss': avg_loss,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    """Evaluate model on validation/test set."""
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []
    
    for batch in tqdm(loader, desc='Evaluating'):
        batch = batch.to(device)
        
        # Prepare node features
        class_embed = model.class_embedding(batch.x_class)
        x = torch.cat([batch.x_bbox, class_embed], dim=-1)
        batch.x = x
        
        # Forward pass
        out = model(batch).squeeze()
        loss = criterion(out, batch.edge_label)
        
        total_loss += loss.item()
        
        # Collect predictions
        probs = torch.sigmoid(out)
        all_preds.extend(probs.cpu().numpy())
        all_labels.extend(batch.edge_label.cpu().numpy())
    
    # Calculate metrics
    avg_loss = total_loss / len(loader)
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    
    pred_binary = (all_preds > 0.5).astype(int)
    accuracy = (pred_binary == all_labels).mean()
    
    tp = ((pred_binary == 1) & (all_labels == 1)).sum()
    fp = ((pred_binary == 1) & (all_labels == 0)).sum()
    fn = ((pred_binary == 0) & (all_labels == 1)).sum()
    tn = ((pred_binary == 0) & (all_labels == 0)).sum()
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    
    return {
        'loss': avg_loss,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'tp': tp, 'fp': fp, 'fn': fn, 'tn': tn
    }


print("Training and evaluation functions defined!")

Training and evaluation functions defined!


In [99]:
# Training loop
best_val_f1 = 0.0
train_losses, train_f1s = [], []
val_losses, val_f1s = [], []

print(f'\\n{"="*60}')
print(f'STARTING GNN TRAINING')
print(f'{"="*60}\\n')

for epoch in range(NUM_EPOCHS):
    print(f'\\n=== Epoch {epoch+1}/{NUM_EPOCHS} ===')
    
    # Train
    train_metrics = train_epoch(model, train_loader, optimizer, criterion, device)
    train_losses.append(train_metrics['loss'])
    train_f1s.append(train_metrics['f1'])
    
    print(f'Train - Loss: {train_metrics["loss"]:.4f}, '
          f'Acc: {train_metrics["accuracy"]:.4f}, '
          f'P: {train_metrics["precision"]:.4f}, '
          f'R: {train_metrics["recall"]:.4f}, '
          f'F1: {train_metrics["f1"]:.4f}')
    
    # Validate
    val_metrics = evaluate(model, valid_loader, criterion, device)
    val_losses.append(val_metrics['loss'])
    val_f1s.append(val_metrics['f1'])
    
    print(f'Valid - Loss: {val_metrics["loss"]:.4f}, '
          f'Acc: {val_metrics["accuracy"]:.4f}, '
          f'P: {val_metrics["precision"]:.4f}, '
          f'R: {val_metrics["recall"]:.4f}, '
          f'F1: {val_metrics["f1"]:.4f}')
    
    # Save best model
    if val_metrics['f1'] > best_val_f1:
        best_val_f1 = val_metrics['f1']
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_f1': best_val_f1,
            'config': {
                'vocab_size': vocab_size,
                'embedding_dim': EMBEDDING_DIM,
                'hidden_dim': HIDDEN_DIM,
                'num_layers': NUM_GNN_LAYERS,
                'dropout': DROPOUT
            }
        }, f'{output_dir}/{exp_name}/model_best.pth')
        print(f'  *** New best validation F1: {best_val_f1:.4f} - Model saved! ***')
    
    # Save checkpoint every 10 epochs
    if (epoch + 1) % 10 == 0:
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
        }, f'{output_dir}/{exp_name}/model_ep{epoch+1}.pth')

print(f'\\n{"="*60}')
print(f'TRAINING COMPLETE')
print(f'Best Validation F1: {best_val_f1:.4f}')
print(f'{"="*60}\\n')

\n============================================================
STARTING GNN TRAINING
============================================================\n
\n=== Epoch 1/50 ===


Training:   0%|          | 0/21 [00:00<?, ?it/s]

Train - Loss: 0.3718, Acc: 0.9448, P: 0.0000, R: 0.0000, F1: 0.0000


Evaluating:   0%|          | 0/7 [00:00<?, ?it/s]

Valid - Loss: 0.1338, Acc: 0.9972, P: 0.0000, R: 0.0000, F1: 0.0000
\n=== Epoch 2/50 ===


...

Valid - Loss: 0.1573, Acc: 0.9972, P: 0.0000, R: 0.0000, F1: 0.0000
\n=== Epoch 9/50 ===


Training:   0%|          | 0/21 [00:00<?, ?it/s]

Train - Loss: 0.2536, Acc: 0.9445, P: 0.3225, R: 0.0042, F1: 0.0082


Evaluating:   0%|          | 0/7 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss plot
axes[0].plot(train_losses, label='Train', marker='o', linewidth=2)
axes[0].plot(val_losses, label='Validation', marker='s', linewidth=2)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title('Training and Validation Loss', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# F1 Score plot
axes[1].plot(train_f1s, label='Train', marker='o', linewidth=2, color='green')
axes[1].plot(val_f1s, label='Validation', marker='s', linewidth=2, color='darkgreen')
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('F1 Score', fontsize=12)
axes[1].set_title('Training and Validation F1 Score', fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{output_dir}/{exp_name}/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Plots saved to {output_dir}/{exp_name}/training_curves.png')